# 04 — ارزیابی عملکرد چت‌بات

In [ ]:
import sys, os, json, time
sys.path.append(os.path.abspath("."))
from chatbot_common import load_intents, load_schema, load_raw_examples, bio_slots_to_dict, VAL_DIR, INTENT_MAPPING_CSV


## بخش ۱ — ارزیابی کمّی روی Validation 


In [ ]:
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
try:
    classify_intent
    extract_all_slots
except NameError:
    print("لطفاً قبل از اجرای این سلول، توابع classify_intent و extract_all_slots را "
          "(با اجرای نوت‌بوک‌های 01 و 02، یا با %run) در دسترس این نوت‌بوک قرار دهید.")


In [ ]:
SAMPLE_SIZE = 100

raw_val = load_raw_examples(VAL_DIR)

if raw_val:
    intent_code_to_name = {}
    if os.path.exists(INTENT_MAPPING_CSV):
        mdf = pd.read_csv(INTENT_MAPPING_CSV)
        intent_code_to_name = dict(zip(mdf["code"], mdf["intent"]))

    examples = raw_val[:SAMPLE_SIZE]

    y_true_intent, y_pred_intent = [], []
    slot_tp = slot_fp = slot_fn = 0
    value_correct = value_total = 0
    latencies = []

    for ex in tqdm(examples):
        gold_intent = intent_code_to_name.get(ex.get("intent_id"), ex.get("intent_id"))
        gold_slots = bio_slots_to_dict(ex["input_text"], ex.get("slots", []))

        t0 = time.time()
        pred_intent = classify_intent(ex["input_text"])
        pred_slots = extract_all_slots(pred_intent, ex["input_text"])
        latencies.append(time.time() - t0)

        y_true_intent.append(gold_intent)
        y_pred_intent.append(pred_intent)

        gk, pk = set(gold_slots), set(pred_slots)
        slot_tp += len(gk & pk)
        slot_fp += len(pk - gk)
        slot_fn += len(gk - pk)
        for k in gk & pk:
            value_total += 1
            if str(gold_slots[k]).strip() == str(pred_slots[k]).strip():
                value_correct += 1

    print("== Intent ==")
    print("Accuracy:", accuracy_score(y_true_intent, y_pred_intent))
    print(classification_report(y_true_intent, y_pred_intent, zero_division=0))

    p = slot_tp / (slot_tp + slot_fp) if (slot_tp + slot_fp) else 0.0
    r = slot_tp / (slot_tp + slot_fn) if (slot_tp + slot_fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    print("== Slot name detection ==")
    print(f"Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}")

    print("== Slot value accuracy (exact match, on correctly-detected slots) ==")
    print(f"{value_correct}/{value_total} = {value_correct / value_total if value_total else 0:.3f}")

    print("== Latency ==")
    print(f"میانگین زمان هر نمونه: {sum(latencies)/len(latencies):.2f} ثانیه")
else:
    print("داده‌ای پیدا نشد .")


## بخش ۲ — ارزیابی کیفی با مکالمات شبیه‌سازی‌شده

In [ ]:
# در صورت نیاز، کلاس MultiTurnChatBot را از نوت‌بوک 03 وارد کنید:
# %run 03_multi_turn_chatbot.ipynb

scenarios = {
    "استعلام موجودی": [
        "موجودی حسابم رو می‌خوام",
        "امروز ساعت ۹ صبح",
    ],
    "پرداخت قبض": [
        "می‌خوام قبض برق رو پرداخت کنم",
        "شناسه قبض ۱۲۳۴۵۶۷۸۹۰",
        "شناسه پرداخت ۹۸۷۶۵۴۳۲۱",
        "۰۹۱۲۳۴۵۶۷۸۹",
        "۱۴۵۳۶",
    ],
    "انتقال پایا": [
        "می‌خوام یک میلیون تومان پایا کنم",
        "علی",
        "محمدی",
        "فردا ساعت ۱۰",
        "بابت اجاره",
        "1234",
        "نه دوره‌ای نیست",
        "IR123456789012345678901234",
        "بانک ملی",
    ],
}

results = {}
for name, turns in scenarios.items():
    print(f"\n\n===== سناریو: {name} =====")
    bot = simulate_conversation(turns)
    results[name] = json.loads(bot.generate_json_response()) if bot.state == "completed" else None

results
